# AMICI quick start in scviva-tools

This notebook adapts the upstream AMICI quick-start tutorial to the `scviva.external.AMICI` wrapper.

The active cells use the current scviva Phase 2A surface: setup, training, predictions, residuals, attention patterns, and neighbor embeddings. Upstream ablation, counterfactual attention, explained-variance, and plotting modules are kept as future-capability notes until those APIs are ported.


In [ ]:
!pip install --quiet scviva-tools


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import torch

import scviva
from scviva.external import AMICI

scviva.settings.seed = 0
torch.manual_seed(0)


## Load spatial transcriptomics data

The upstream tutorial uses a mouse cortex h5ad from Figshare. The current scviva wrapper trains and infers on the registered AnnData object, so this tutorial keeps the train split as the registered object.


In [ ]:
adata_path = "./data/mouse_cortex_tutorial.h5ad"
adata = sc.read(adata_path, backup_url="https://figshare.com/ndownloader/files/58303438")
adata.obsm["spatial"] = adata.obs[["centroid_x", "centroid_y"]].to_numpy()

labels_key = "subclass"
if "in_test" not in adata.obs:
    rng = np.random.default_rng(0)
    adata.obs["in_test"] = rng.random(adata.n_obs) < 0.2

adata_train = adata[~adata.obs["in_test"].astype(bool)].copy()
print("Full dataset:", adata.shape)
print("Training subset:", adata_train.shape)


In [ ]:
plot_df = pd.DataFrame(adata.obsm["spatial"], columns=["x", "y"], index=adata.obs_names)
plot_df[labels_key] = adata.obs[labels_key].values
plot_df["in_test"] = adata.obs["in_test"].astype(bool).values

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=plot_df, x="x", y="y", hue=labels_key, s=8, linewidth=0, ax=ax)
ax.set_title("Mouse cortex spatial labels")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.show()


## Setup and train AMICI

AMICI registers expression, labels, spatial coordinates, and label-aware spatial neighbors through `AMICI.setup_anndata`. The compact scviva module exposes a smaller parameter set than upstream AMICI.


In [ ]:
exp_params = {
    "n_neighbors": 30,
    "epochs": 50,
    "batch_size": 512,
    "lr": 1e-3,
}
model_params = {
    "n_label_embed": 16,
    "n_nn_embed": 64,
    "n_hidden": 128,
}

device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
AMICI.setup_anndata(
    adata_train,
    labels_key=labels_key,
    spatial_key="spatial",
    n_neighbors=exp_params["n_neighbors"],
)
model = AMICI(adata_train, **model_params)


In [ ]:
model.train(
    max_epochs=exp_params["epochs"],
    batch_size=exp_params["batch_size"],
    lr=exp_params["lr"],
    device=device,
    prog_bar=True,
)


## Retrieve Phase 2A outputs

The current scviva wrapper can write tutorial-friendly arrays back to the registered AnnData. Prediction and residual arrays are stored in `obsm`; attention patterns are one row per cell and one column per neighbor; neighbor embeddings are stored as an observation-aligned tensor in `obsm`.


In [ ]:
predictions = model.get_predictions(batch_size=128, store_key="amici_prediction")
residuals = model.get_predictions(batch_size=128, get_residuals=True, store_key="amici_residual")
attention = model.get_attention_patterns(batch_size=128, store_key="amici_attention")
neighbor_embeddings = model.get_nn_embed(batch_size=128, store_key="X_amici_nn")

print("predictions:", predictions.shape)
print("residuals:", residuals.shape)
print("attention:", attention.shape)
print("neighbor embeddings:", neighbor_embeddings.shape)


In [ ]:
x = adata_train.X.toarray() if hasattr(adata_train.X, "toarray") else np.asarray(adata_train.X)
adata_train.obs["amici_reconstruction_mse"] = np.mean((predictions - x) ** 2, axis=1)
adata_train.obs["amici_max_attention"] = attention.max(axis=1)

sc.pl.embedding(
    adata_train,
    basis="spatial",
    color=[labels_key, "amici_reconstruction_mse", "amici_max_attention"],
    frameon=False,
    ncols=3,
)


## Upstream interpretation sections not yet ported

The upstream AMICI tutorial continues with high-level interaction scores, per-gene ablation, counterfactual attention, and empirical attention plotting. Those APIs are planned for later AMICI phases in scviva-tools, so they are intentionally not active code here.


In [ ]:
# TODO: Phase 5 — AMICI interpretation modules
# Enable this section after the following APIs are ported into scviva.external.amici:
# - AMICIAblationModule
# - AMICICounterfactualAttentionModule
# - AMICIAttentionModule
# - explained-variance and plotting helpers
#
# Upstream reference only:
#   ablation_residuals = model.get_neighbor_ablation_scores(adata=adata_train)
#   attention_patterns = model.get_attention_patterns(adata_train)
#   counterfactual = model.get_counterfactual_attention_patterns(cell_type="Astro", adata=adata_train)
